# Data Wrangling 2.3 Solutions

In [ ]:
import math
import numpy as np
import pandas as pd

import psycopg2

import json

import csv

from datetime import datetime as dt

from IPython.display import display, HTML


In [ ]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [ ]:
cursor = connection.cursor()

In [ ]:
#
# function to run a select query and return rows in a pandas dataframe
# pandas puts all numeric values from postgres to float
# if it will fit in an integer, change it to integer
#

def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)
    

## You try it - see if the product_id in stage_3_line_items has valid numeric data

In [ ]:
connection.rollback()

query = """

select product_id::numeric
from stage_3_line_items


"""
cursor.execute(query)

connection.rollback()

## You try it - find customer_id's in the stage_3_sales table that are not in the stage_3_customers table

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query  = """

select * 
from stage_3_sales 
where customer_id not in (select customer_id from stage_3_customers)


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

## You try it - find store_id's in the stage_3_sales table that are not in the stores table

In [ ]:
rollback_before_flag = True
rollback_after_flag = True

query  = """

select *
from stage_3_sales
where store_id::numeric not in (select store_id from stores)


"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)